# Day 2-03｜Roboflow BBOX Dataset 與 YOLO Detection 訓練
> Python 籃球運動資料分析課程  
> 本單元使用 Day 1 BBOX 作業的 Roboflow YOLO export 格式，提供可執行的 Ultralytics 訓練程式，並用已訓練模型對參考影片輸出 preview。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 確認 Roboflow detection dataset 的 `data.yaml` 與 images/labels 結構。
- 閱讀可執行的 YOLO detection 訓練程式。
- 使用已訓練 detector 對參考影片產生 preview。


## 資料放置位置
- Day 1 BBOX 作業 Roboflow YOLO export：`assets/datasets/roboflow_bbox_yolo/`
- 已訓練 detector 權重：`assets/models/detectors/yolo26n_basketball_player_best.pt`
- 訓練輸出：`assets/results/training/bbox_detection/`
- Preview 輸出：`assets/results/d2_03_trained_detector_preview.mp4`


In [ ]:
from pathlib import Path
import sys

# Colab 重新啟動 runtime 後，先掛載學生自己的 Google Drive。
try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
    drive.mount("/content/drive")
except ModuleNotFoundError:
    IN_COLAB = False

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機驗證或 zip 解壓後執行時，從目前資料夾往上找課程根目錄。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if not COURSE_ROOT.exists():
    raise FileNotFoundError("找不到課程資料夾，請先執行 init_colab.ipynb。")

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from src.course_setup import install_requirements_if_colab, print_environment_summary  # noqa: E402

install_requirements_if_colab(COURSE_ROOT)
print_environment_summary(COURSE_ROOT)


In [ ]:
from src.roboflow_utils import dataset_status

BBOX_DATASET_DIR = COURSE_ROOT / "assets" / "datasets" / "roboflow_bbox_yolo"
MODEL_PATH = COURSE_ROOT / "assets" / "models" / "detectors" / "yolo26n_basketball_player_best.pt"

print(dataset_status(BBOX_DATASET_DIR))
print("model exists:", MODEL_PATH.exists(), MODEL_PATH)


In [ ]:
RUN_DOWNLOAD_FROM_ROBOFLOW = False

if RUN_DOWNLOAD_FROM_ROBOFLOW:
    from roboflow import Roboflow

    rf = Roboflow(api_key="YOUR_API_KEY")
    project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
    version = project.version(1)
    dataset = version.download("yolov8", location=str(BBOX_DATASET_DIR))
    print(dataset.location)
else:
    print("RUN_DOWNLOAD_FROM_ROBOFLOW=False；請手動匯出或在需要時填入 Roboflow API 設定。")


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    from ultralytics import YOLO

    data_yaml = BBOX_DATASET_DIR / "data.yaml"
    if not data_yaml.exists():
        raise FileNotFoundError(
            "找不到 data.yaml。請先將 Roboflow YOLO detection export 放到 assets/datasets/roboflow_bbox_yolo/"
        )

    model = YOLO("yolo26n.pt")
    results = model.train(
        data=str(data_yaml),
        epochs=100,
        imgsz=960,
        batch=2,
        workers=4,
        patience=30,
        close_mosaic=10,
        project=str(COURSE_ROOT / "assets" / "results" / "training" / "bbox_detection"),
        name="yolo26n_basketball_player",
        plots=True,
    )
    print(results)
else:
    print("RUN_TRAINING=False；課堂預設使用已訓練 detector 權重進行推論。")


In [ ]:
from src.yolo_utils import first_reference_video, write_detection_preview_video

video_path = first_reference_video(COURSE_ROOT)
preview_path, records = write_detection_preview_video(
    video_path=video_path,
    model_path=MODEL_PATH,
    output_path=COURSE_ROOT / "assets" / "results" / "d2_03_trained_detector_preview.mp4",
    max_frames=120,
    conf=0.25,
    imgsz=960,
)

print("video:", video_path)
print("preview video:", preview_path)
print("preview json:", preview_path.with_suffix(".json"))
print("records:", len(records))
